# CUT (HAT 기반) 실험 — 학습 + 평가 (직접 실행)

**도메인 시프트 해소판.** 리파이너 입력 = 팀장님의 **진짜 HAT-SR 출력**(RRDB 근사 아님).
한 커널에서 CUT 학습 → refined 결과 → `SR-HAT / CycleGAN / CUT` 비교.

- 커널 `trellis` → Restart Kernel → Run All
- 데이터 `refine_hat` (HAT-SR→HR, train 420 / test 60, 512)
- ⚠️ 셀 2(학습)은 35 epoch ≈ **2시간+**. 빨리 보려면 `--n_epochs` 줄이기. VS Code 닫으면 커널 죽음(오래 비울 거면 터미널 `bash scripts/run_cut_hat.sh`).
- 참고(팀장 전체셋): SR-HAT 14.85 / CycleGAN-on-HAT 13.00


## 0. 환경 설정

In [ ]:
import os, sys, time, glob
import numpy as np
from PIL import Image
import torch
import matplotlib.pyplot as plt
ROOT = os.path.expanduser("~/erang_sr")
CUT  = os.path.join(ROOT, "refiners/cut_src")
os.chdir(CUT)
sys.path.insert(0, CUT)
for p in (ROOT, os.path.join(ROOT, "team_aiduo")):
    if p not in sys.path: sys.path.append(p)
DEV = "cuda"
def load01(p):
    im = np.asarray(Image.open(p).convert("RGB")).astype(np.float32)/255.0
    return torch.from_numpy(im.transpose(2,0,1)).unsqueeze(0).float()
def save01(t, p):
    a = (t.detach().cpu().clamp(0,1).numpy().transpose(1,2,0)*255).round().astype("uint8")
    Image.fromarray(a).save(p)
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


## 1. 데이터 확인 (HAT-SR → HR)
`refine_hat`: 리파이너 입력이 **HAT ×4 실제 SR 출력**(팀장 sr_outputs), 타겟은 HR.

In [ ]:
DR = os.path.join(ROOT, "refiners/data/refine_hat")
for s in ["trainA","trainB","testA","testB"]:
    print(f"  {s}: {len(os.listdir(os.path.join(DR,s)))}장")
a = sorted(glob.glob(DR+"/testA/*.png"))[0]; b = a.replace("testA","testB")
fig, ax = plt.subplots(1,2,figsize=(8,4))
ax[0].imshow(Image.open(a)); ax[0].set_title("HAT-SR (input, 팀장 출력)"); ax[0].axis("off")
ax[1].imshow(Image.open(b)); ax[1].set_title("HR (target/GT)"); ax[1].axis("off")
plt.tight_layout(); plt.show()


## 2. CUT 학습 (HAT-SR 도메인, loss 관찰)
입력이 HAT-SR이라 CUT이 이 분포를 직접 학습 → 도메인 시프트 없음.

In [ ]:
sys.argv = ["train.py", "--dataroot", DR, "--name", "cut_hat", "--CUT_mode", "CUT",
    "--display_id", "0", "--gpu_ids", "0", "--batch_size", "1",
    "--n_epochs", "25", "--n_epochs_decay", "10",      # <- 빨리 보려면 줄이기 (예: 5,0)
    "--load_size", "512", "--crop_size", "256", "--print_freq", "200"]
from options.train_options import TrainOptions
from data import create_dataset
from models import create_model
opt = TrainOptions().parse(); opt.num_threads = 0
dataset = create_dataset(opt); model = create_model(opt)
print(f"\n학습 {len(dataset)}장 · {opt.n_epochs+opt.n_epochs_decay} epoch 시작\n")
t0 = time.time(); step = 0
for epoch in range(opt.epoch_count, opt.n_epochs + opt.n_epochs_decay + 1):
    for i, data in enumerate(dataset):
        if epoch == opt.epoch_count and i == 0:
            model.data_dependent_initialize(data); model.setup(opt); model.parallelize()
        model.set_input(data); model.optimize_parameters(); step += 1
        if step % 50 == 0:
            L = model.get_current_losses()
            print(f"ep{epoch} step{step:>5}  G_GAN {L['G_GAN']:.3f}  NCE {L['NCE']:.3f}  "
                  f"D_real {L['D_real']:.3f}  ({time.time()-t0:.0f}s)")
    print(f"--- epoch {epoch} done ({time.time()-t0:.0f}s) ---")
print(f"\n학습 종료 ({time.time()-t0:.0f}s)")


## 3. 학습된 CUT의 refined 결과 (테스트 3장)

In [ ]:
G_cut = model.netG; G_cut.eval()
def refine(G, x):
    with torch.no_grad(): return ((G(x*2-1)+1)/2).clamp(0,1)
tests = sorted(glob.glob(DR+"/testA/*.png"))[:3]
fig, ax = plt.subplots(len(tests), 3, figsize=(11, 3.6*len(tests)))
ax = ax.reshape(len(tests), 3)
for r, ta in enumerate(tests):
    x = load01(ta).to(DEV); ref = refine(G_cut, x)
    imgs = [x[0].cpu(), ref[0].cpu(), load01(ta.replace("testA","testB"))[0]]
    for c,(im,t) in enumerate(zip(imgs, ["HAT-SR (input)","CUT refined","HR (GT)"])):
        ax[r,c].imshow(im.numpy().transpose(1,2,0)); ax[r,c].axis("off")
        if r==0: ax[r,c].set_title(t)
plt.tight_layout(); plt.show()


## 4. Dual-Track 평가 — SR-HAT vs CycleGAN vs CUT (HAT 입력 기준)

In [ ]:
from cycleGen_model import load_cyclegan_model
G_cyc = load_cyclegan_model(os.path.join(ROOT,"weights/best_G_AB.pth"), device=DEV)
allA = sorted(glob.glob(DR+"/trainA/*.png")) + sorted(glob.glob(DR+"/testA/*.png"))
test_names = set(os.path.basename(p) for p in glob.glob(DR+"/testA/*.png"))
base = os.path.join(ROOT, "runs/eval_hat")
dirs = {k: os.path.join(base,k) for k in ["sr_hat","cyclegan","cut"]}
hr_ref = os.path.join(base,"hr_ref")
for dd in list(dirs.values())+[hr_ref]: os.makedirs(dd, exist_ok=True)
print(f"{len(allA)}장 처리 중...")
for j, ta in enumerate(allA):
    name = os.path.basename(ta); x = load01(ta).to(DEV)
    save01(x[0], os.path.join(dirs["sr_hat"], name))              # SR-HAT = 입력 그대로
    save01(refine(G_cyc, x)[0], os.path.join(dirs["cyclegan"], name))
    save01(refine(G_cut, x)[0], os.path.join(dirs["cut"], name))
    save01(load01(ta.replace("/trainA/","/trainB/").replace("/testA/","/testB/"))[0],
           os.path.join(hr_ref, name))
    if (j+1) % 100 == 0: print(f"  {j+1}/{len(allA)}")
print("처리 완료 · 지표 계산...")
from skimage.metrics import structural_similarity as ssim
import lpips as L, pyiqa
from pytorch_fid.fid_score import calculate_fid_given_paths
lp = L.LPIPS(net="alex").to(DEV); niqe = pyiqa.create_metric("niqe", device=DEV)
def psnr(a,b):
    mse=float(((a-b)**2).mean()); return 99.0 if mse==0 else 10*np.log10(1/mse)
rows = {}
for cond, dd in dirs.items():
    ps,ss,lps = [],[],[]
    for name in sorted(test_names):
        out=load01(os.path.join(dd,name)); hr=load01(os.path.join(hr_ref,name))
        a=out[0].numpy().transpose(1,2,0); b=hr[0].numpy().transpose(1,2,0)
        ps.append(psnr(a,b)); ss.append(ssim(a,b,channel_axis=2,data_range=1.0))
        with torch.no_grad(): lps.append(float(lp(out.to(DEV)*2-1, hr.to(DEV)*2-1).mean()))
    nqs=[float(niqe(os.path.join(dd,n))) for n in os.listdir(dd)]
    try: fid=calculate_fid_given_paths([dd,hr_ref],batch_size=50,device=DEV,dims=2048)
    except Exception as e: fid=float("nan"); print("FID err",cond,e)
    rows[cond]=(np.mean(ps),np.mean(ss),np.mean(lps),fid,np.mean(nqs))
print("\n======= Dual-Track (HAT 입력 기준, test 60장) =======")
print(f"{'조건':10s} | {'PSNR up':>8} {'SSIM up':>8} {'LPIPS dn':>9} | {'FID dn':>8} {'NIQE dn':>8}")
print("-"*62)
for k in ["sr_hat","cyclegan","cut"]:
    p,s,l,f,n=rows[k]; print(f"{k:10s} | {p:8.3f} {s:8.4f} {l:9.4f} | {f:8.3f} {n:8.3f}")
print("\n핵심: cut(CUT-on-HAT)이 cyclegan보다 좋으면 → 실제 파이프라인에서도 CUT 우위 확정.")


## 해석
- 이 `cut` = **HAT-SR로 재학습한 CUT** (도메인 시프트 없음).
- `sr_hat`(입력) 대비 `cyclegan`이 떨어지는지, `cut`이 회복하는지가 핵심.
- 팀장님 전체셋(SR-HAT 14.85, CycleGAN 13.00)과 방향 일치하는지 확인.
